In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/data.csv')

dtype('O')

In [2]:
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])
df['TransactionStartTime'].dtype

datetime64[ns, UTC]

In [3]:
snapshot_date = df['TransactionStartTime'].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2019-02-14 10:01:28+0000', tz='UTC')

In [ ]:
rfm = (
    df.groupby('CustomerId')
    .agg(
        recency=('TransactionStartTime', lambda x: (snapshot_date - x.max()).days),
        frequency=('TransactionId', 'count'),
        monetary=('Amount', 'sum')
    ).reset_index()
)


rfm.describe()

,recency,frequency,monetary
count,3742.000000,3742.000000,3.742000e+03
mean,31.461251,25.564404,1.717377e+05
std,27.118932,96.929602,2.717305e+06
min,1.000000,1.000000,-1.049000e+08
25%,6.000000,2.000000,4.077438e+03
50%,25.000000,7.000000,2.000000e+04
75%,54.000000,20.000000,7.996775e+04
max,91.000000,4091.000000,8.345124e+07


In [7]:
rfm_features = rfm[['recency', 'frequency', 'monetary']]
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

rfm_scaled = pd.DataFrame(
    rfm_scaled, columns=['recency_scaled', 'frequency_scaled', 'monetary_scaled'],
    index=rfm.index
)

rfm_scaled.describe()

,recency_scaled,frequency_scaled,monetary_scaled
count,3.742000e+03,3.742000e+03,3.742000e+03
mean,3.892605e-17,-7.595326e-18,-9.494157e-18
std,1.000134e+00,1.000134e+00,1.000134e+00
min,-1.123397e+00,-2.534591e-01,-3.867280e+01
25%,-9.389991e-01,-2.431409e-01,-6.170920e-02
50%,-2.382879e-01,-1.915502e-01,-5.584873e-02
75%,8.312186e-01,-5.741433e-02,-3.377694e-02
max,2.195761e+00,4.194775e+01,3.065193e+01


In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)


In [12]:
cluster_summary = (
    rfm.groupby('cluster')[['recency', 'frequency', 'monetary']]
    .mean()
    .round(2)
)

cluster_summary


,recency,frequency,monetary
cluster,,,
0,61.88,7.72,8.172068e+04
1,12.73,34.80,2.725740e+05
2,29.00,4091.00,-1.049000e+08


In [14]:
rfm['is_high_risk'] = (rfm['cluster'] == 0).astype(int)
rfm['is_high_risk'].value_counts()

is_high_risk
0    2316
1    1426
Name: count, dtype: int64

In [15]:
rfm[['CustomerId', 'is_high_risk']].to_csv(
    "../data/processed/risk_labels.csv",
    index=False
)